# Miko-Ai — Train LoRA Adapter + Upload HuggingFace

Jalankan di **Kaggle** (GPU T4) atau Colab.

Ganti variabel di cell Setup sebelum Run All.

**Catatan jujur:** notebook ini menyiapkan 1 adapter per run. Ulangi untuk 6 bidang (api, server, web, apk, security, coding).

In [ ]:
# === SETUP — GANTI SESUAI KEBUTUHAN ===
DOMAIN = "api"  # api | server | web | apk | security | coding
HF_TOKEN = "hf_XXX"  # token Write dari huggingface.co/settings/tokens
HF_REPO = f"v1-byte/miko-{DOMAIN}-lora"
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # lebih kecil = lebih mudah di T4
DATA_URLS = [
  # ganti dengan raw.githubusercontent.com/.../dataset_template/api/api_sample_50.jsonl
  # atau upload file ke Kaggle Dataset lalu path lokal
]
MAX_STEPS = 200
LORA_R = 16
print("Domain:", DOMAIN, "| Repo:", HF_REPO)

In [ ]:
!pip install -q transformers peft bitsandbytes trl datasets accelerate huggingface_hub
import os, json, torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from huggingface_hub import HfApi, login
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())

In [ ]:
# Load dataset JSONL format {"user","assistant"}
# Opsi A: file lokal di Kaggle
local_files = [p for p in [
  f"/kaggle/input/miko-data/{DOMAIN}_sample.jsonl",
  f"./dataset_template/{DOMAIN}/{DOMAIN}_sample_30.jsonl",
  f"./dataset_template/{DOMAIN}/{DOMAIN}_sample_50.jsonl",
] if os.path.exists(p)]
rows = []
for path in local_files:
  with open(path) as f:
    for line in f:
      line=line.strip()
      if not line: continue
      o=json.loads(line)
      if o.get("user") and o.get("assistant"):
        rows.append({"text": f"### User:\n{o['user']}\n\n### Assistant:\n{o['assistant']}"})
print("samples", len(rows))
if len(rows) < 5:
  raise SystemExit("Dataset terlalu kecil. Upload JSONL dulu atau set DATA_URLS.")
ds = Dataset.from_list(rows)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
  BASE_MODEL,
  load_in_4bit=True,
  device_map="auto",
  trust_remote_code=True,
)
lora = LoraConfig(
  r=LORA_R, lora_alpha=32, lora_dropout=0.05,
  target_modules=["q_proj","k_proj","v_proj","o_proj"],
  bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
  output_dir=f"./out-miko-{DOMAIN}",
  per_device_train_batch_size=2,
  gradient_accumulation_steps=4,
  learning_rate=2e-4,
  max_steps=MAX_STEPS,
  logging_steps=10,
  save_steps=MAX_STEPS,
  fp16=True,
  report_to=[],
)
trainer = SFTTrainer(
  model=model,
  train_dataset=ds,
  args=args,
  processing_class=tokenizer,
)
trainer.train()
out_dir = f"miko-{DOMAIN}-lora"
trainer.model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print("Saved", out_dir)

In [ ]:
# Upload ke HuggingFace (butuh HF_TOKEN Write)
if not HF_TOKEN or HF_TOKEN.startswith("hf_XXX"):
  print("Skip upload: set HF_TOKEN dulu")
else:
  login(token=HF_TOKEN)
  api = HfApi()
  api.create_repo(HF_REPO, private=False, exist_ok=True, repo_type="model")
  api.upload_folder(folder_path=out_dir, repo_id=HF_REPO, repo_type="model")
  print("Uploaded https://huggingface.co/" + HF_REPO)

## Setelah upload
1. Di Cloudflare Worker set secret `HF_TOKEN` dan env `MIKO_MODEL_API=v1-byte/miko-api-lora` (dst).
2. Atau untuk inference API lama: pastikan model support `api-inference`.
3. Ulangi notebook dengan `DOMAIN` berbeda sampai 6 adapter siap.
4. Dataset ideal: 500–2000 Q&A **berkualitas** per bidang, bukan volume kosong.